In [ ]:
import numpy as np
import openpyxl
from datetime import date

data_path = '/workspace/data/MO14-Time-is-Money-Data.xlsx'
wb = openpyxl.load_workbook(data_path, data_only=True)
ws = wb['Data']

N = 10  # 10-year forecast (Years 1-10 = 2015-2024)
# Columns: 7=Year0(2014), 8=Year1(2015), ..., 17=Year10(2024)

def read_row(row, start=8, count=N):
    return [ws.cell(row, c).value or 0 for c in range(start, start + count)]

# Revenue, Opex, Capex in real USD (rows 12-15)
rev_real = np.array(read_row(12), dtype=float)
opex_real = np.array(read_row(13), dtype=float)
capex_a_real = np.array(read_row(14), dtype=float)
capex_b_real = np.array(read_row(15), dtype=float)

# Opening tax assets (Year 0, col 7)
open_a = float(ws.cell(18, 7).value)  # 700
open_b = float(ws.cell(19, 7).value)  # 450
life_a = int(ws.cell(21, 7).value)    # 5
life_b = int(ws.cell(22, 7).value)    # 4
tax_rate = float(ws.cell(25, 7).value)  # 0.4
disc_rate = float(ws.cell(37, 7).value)  # 0.1
spot_fx = float(ws.cell(34, 7).value)   # 1.31

# Inflation
us_inf = np.array(read_row(30), dtype=float)
eu_inf = np.array(read_row(31), dtype=float)

# Real FX (USD per EUR)
real_fx = np.array(read_row(34), dtype=float)

wb.close()

# Cumulative inflation indices (from base Year 0)
cum_us = np.cumprod(1 + us_inf)
cum_eu = np.cumprod(1 + eu_inf)

# Nominal FX: nom_FX_t = real_FX_t * cum_US_t / cum_EU_t
nom_fx = real_fx * cum_us / cum_eu

print(f"Open A={open_a}, B={open_b}, Life A={life_a}, B={life_b}")
print(f"Tax={tax_rate}, Disc={disc_rate}, Spot={spot_fx}")
print(f"Nominal FX: {np.round(nom_fx, 4)}")

In [ ]:
# Nominal values
rev_nom_usd = rev_real * cum_us
opex_nom_usd = opex_real * cum_us
capex_a_nom = capex_a_real * cum_us
capex_b_nom = capex_b_real * cum_us
total_capex_nom = capex_a_nom + capex_b_nom

# Pure DDB depreciation (no switch to SL - continues indefinitely)
def ddb_pure(opening, life, n):
    rate = 2.0 / life
    dep = np.zeros(n)
    bv = opening
    for y in range(n):
        d = rate * bv
        dep[y] = d
        bv -= d
    return dep

# Class A opening balance depreciation (starts Year 1)
dep_a = ddb_pure(open_a, life_a, N)

# Class A capex vintages: capex at end of year t, dep starts year t+1
for t in range(N):
    if capex_a_nom[t] > 0 and t + 1 < N:
        v = ddb_pure(capex_a_nom[t], life_a, N - t - 1)
        for j in range(len(v)):
            dep_a[t + 1 + j] += v[j]

# Class B opening balance: SL over 4 years starting Year 1
dep_b = np.zeros(N)
for y in range(min(life_b, N)):
    dep_b[y] += open_b / life_b

# Class B capex vintages: SL over 4 years, starts year after purchase
for t in range(N):
    if capex_b_nom[t] > 0 and t + 1 < N:
        annual = capex_b_nom[t] / life_b
        for y in range(min(life_b, N - t - 1)):
            dep_b[t + 1 + y] += annual

total_dep_usd = dep_a + dep_b

# Convert to nominal EUR
spot_fx = 1.31
rev_nom_eur = rev_nom_usd / nom_fx
dep_nom_eur = total_dep_usd / nom_fx

# P&L in nominal USD
npbt_usd = rev_nom_usd - opex_nom_usd - total_dep_usd
tax_usd = np.maximum(npbt_usd * tax_rate, 0)
npbt_eur = npbt_usd / nom_fx
tax_eur = tax_usd / nom_fx

# FCF in nominal USD (pre-financing, post-tax)
fcf_usd = rev_nom_usd - opex_nom_usd - tax_usd - total_capex_nom

# NPV and XNPV computed in USD (10% rate is for USD CFs), then convert to EUR
npv_usd = sum(fcf_usd[i] / (1 + disc_rate)**(i+1) for i in range(N))
npv_eur = npv_usd / spot_fx

val_date = date(2014, 12, 31)
cf_dates = [date(2015 + i, 12, 31) for i in range(N)]
xnpv_usd = sum(fcf_usd[i] / (1 + disc_rate)**((cf_dates[i] - val_date).days / 365.0) for i in range(N))
xnpv_eur = xnpv_usd / spot_fx

pv_decrease_eur = (npv_eur - xnpv_eur) * 1e6

print(f"Nom FX Year 7: {nom_fx[6]:.4f}")
print(f"Revenue Year 2 (nom EUR): {rev_nom_eur[1]:.1f}m")
print(f"Dep Year 6 (nom EUR): {dep_nom_eur[5]:.1f}m")
print(f"NPBT Year 4 (nom EUR): {npbt_eur[3]:.1f}m")
print(f"Tax Year 8 (nom EUR): {tax_eur[7]:.1f}m")
print(f"NPV (nom EUR): {npv_eur:.1f}m")
print(f"PV decrease: {pv_decrease_eur:.0f} EUR")

In [ ]:
# Match to multiple choice and build answers

def closest(opts, val):
    return min(opts, key=lambda k: abs(opts[k] - val))

q1 = closest({'A': 1.297, 'B': 1.308, 'C': 1.312, 'D': 1.324}, nom_fx[6])
q2 = closest({'A': 740, 'B': 745, 'C': 1214, 'D': 1220}, rev_nom_eur[1])
q3 = closest({'A': 268.3, 'B': 269.0, 'C': 270.7, 'D': 271.5}, dep_nom_eur[5])
q4 = closest({'A': 44.4, 'B': 44.8, 'C': 45.2, 'D': 45.6}, npbt_eur[3])
q5 = closest({'A': 76, 'B': 78, 'C': 80, 'D': 82}, tax_eur[7])
q6 = closest({'A': 515, 'B': 519, 'C': 522, 'D': 523}, npv_eur)
q7 = closest({'A': 179000, 'B': 180000, 'C': 181000, 'D': 182000}, pv_decrease_eur)

answers = {
    'question1': q1,
    'question2': q2,
    'question3': q3,
    'question4': q4,
    'question5': q5,
    'question6': q6,
    'question7': q7,
}
print("answers =", answers)